# 07 — Hyperparameter Tuning

Tune the champion model using RandomizedSearchCV, then save the tuned artifact.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import joblib, yaml, os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','axes.labelcolor':'#e2e8f0','xtick.color':'#94a3b8',
    'ytick.color':'#94a3b8','text.color':'#e2e8f0','grid.color':'#334155'})
PALETTE = ['#38bdf8','#fb7185','#34d399','#fbbf24','#a78bfa','#f97316']

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../configs/model.yaml') as f:
    mcfg = yaml.safe_load(f)

In [ ]:
# ── Load data and split ───────────────────────────────────────────────────────
df = pd.read_csv(f'../{paths["data"]["processed"]}data_engineered.csv')
TARGET = cfg['project']['target']
X = df.drop(columns=[TARGET])
y = df[TARGET]

TEST_SIZE = cfg['project']['test_size']
split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── Determine champion from saved model ───────────────────────────────────────
# We default to tuning Random Forest. Change `CHAMPION` if your NB-06 results differ.
CHAMPION = 'random_forest'  # options: random_forest | xgboost | lightgbm | catboost | gradient_boosting
print(f'Tuning: {CHAMPION}')

In [ ]:
# ── Param grids ───────────────────────────────────────────────────────────────
param_grids = {
    'random_forest': {
        'n_estimators':    [100, 200, 300, 500],
        'max_depth':       [None, 6, 8, 10, 15],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
        'max_features':    ['sqrt', 'log2', 0.5],
    },
    'xgboost': {
        'n_estimators':   [200, 300, 500],
        'learning_rate':  [0.01, 0.05, 0.1],
        'max_depth':      [4, 6, 8],
        'subsample':      [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':          [0, 0.1, 0.5],
    },
    'lightgbm': {
        'n_estimators':  [200, 300, 500],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves':    [20, 31, 50, 100],
        'max_depth':     [-1, 6, 10],
        'min_child_samples': [5, 10, 20],
    },
    'gradient_boosting': {
        'n_estimators':  [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth':     [3, 5, 7],
        'subsample':     [0.6, 0.8, 1.0],
        'min_samples_split': [2, 5, 10],
    },
}

base_models = {
    'random_forest':     RandomForestRegressor(random_state=42, n_jobs=-1),
    'xgboost':           xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    'lightgbm':          lgb.LGBMRegressor(random_state=42, verbose=-1, n_jobs=-1),
    'gradient_boosting': GradientBoostingRegressor(random_state=42),
}

base_model = base_models[CHAMPION]
param_grid = param_grids[CHAMPION]
print('Param grid:', param_grid)

In [ ]:
# ── Run RandomizedSearchCV ────────────────────────────────────────────────────
RS = mcfg['tuning']['random_state']
N_ITER = mcfg['tuning']['n_iter']
CV = mcfg['tuning']['cv']

search = RandomizedSearchCV(
    base_model, param_grid,
    n_iter=N_ITER, cv=CV,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, random_state=RS, verbose=1,
    return_train_score=True
)
search.fit(X_train, y_train)

print(f'\nBest CV RMSE: {-search.best_score_:.2f}')
print('Best params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Evaluate tuned model on test set ─────────────────────────────────────────
best_model = search.best_estimator_
pred_test  = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred_test))
mae  = mean_absolute_error(y_test, pred_test)
r2   = r2_score(y_test, pred_test)

print(f'\nTuned Model — Test Metrics')
print(f'  R²  : {r2:.4f}')
print(f'  RMSE: {rmse:.2f}')
print(f'  MAE : {mae:.2f}')

In [ ]:
# ── CV results visualisation ──────────────────────────────────────────────────
cv_res = pd.DataFrame(search.cv_results_)
cv_res['rmse'] = -cv_res['mean_test_score']
cv_res_sorted = cv_res.sort_values('rmse').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cv_res_sorted.index, cv_res_sorted['rmse'], color=PALETTE[0], linewidth=2)
ax.fill_between(cv_res_sorted.index,
    cv_res_sorted['rmse'] - cv_res_sorted['std_test_score'].abs(),
    cv_res_sorted['rmse'] + cv_res_sorted['std_test_score'].abs(),
    alpha=0.2, color=PALETTE[0])
ax.axhline(rmse, color=PALETTE[1], linestyle='--', label=f'Test RMSE={rmse:.1f}')
ax.set(title=f'RandomizedSearchCV Results — {CHAMPION}',
       xlabel='Candidate rank (sorted by CV RMSE)', ylabel='CV RMSE')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/tuning_cv_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save tuned model ──────────────────────────────────────────────────────────
model_path = f'../{paths["artifacts"]["models"]}tuned_model.pkl'
joblib.dump(best_model, model_path)
print(f'Tuned model saved → {model_path}')

import json
params_path = f'../{paths["artifacts"]["models"]}best_params.json'
with open(params_path, 'w') as f:
    json.dump(search.best_params_, f, indent=2, default=str)
print(f'Best params saved → {params_path}')

## Tuning Summary

| Step | Detail |
|---|---|
| Method | RandomizedSearchCV |
| CV folds | 5 |
| Iterations | 50 |
| Scoring | neg_RMSE |

**Next:** `08_Model_Evaluation.ipynb`
